

Name : Nencyben Vijaybhai Rathod

Student ID : GH1036105

Assessment Title : Individual Project

Module Code : SSO626 Retake

Module Title : M515 Ethical Issues for AI

Date Submitted : 18 september 2026


# Fairness Analysis and Bias Mitigation in Bank Marketing Prediction


GitHub Link :

https://github.com/NencyRathod5418/M515-Ethical-Issues-for-AI-Fairness-Analysis

Dataset:

UCI Bank Marketing Dataset (Bank Marketing, `bank-full.csv`)
https://archive.ics.uci.edu/dataset/222/bank+marketing
[link text](https://)





# 1. Problem Statement and Ethical Context

The purpose of using the Bank Marketing dataset is to demonstrate the possibility of using machine learning for predicting the probability of subscribing to bank term deposits following a marketing campaign. This dataset contains 45,211 observations and 17 features, including demographics, financial, and marketing-related characteristics. The feature y is the indicator that states whether or not the customer subscribes to the term deposit.

The baseline Machine Learning approach used is the Random Forest Classification algorithm. It is needed to measure predictive performance and to find out if the results or errors produced by this algorithm differ depending on the age groups. Age is the protected attribute and is divided into two categories: Younger customers (<40) and Older customers (40+).

The problem from the ethical standpoint is that such prediction algorithms can result in discrimination or differential error rates between different demographic groups in their application within a financial marketing environment. In addition to model accuracy, then, there are other factors to be considered, which include the selection rates and such fairness indicators as Demographic Parity Difference (DPD), Disparate Impact (DI), True Positive Rate (TPR), False Positive Rate (FPR), False Negative Rate (FNR), False Discovery Rate (FDR) and False Omission Rate (FOR).

Another important distinction is made between disparate treatment and disparate mistreatment. Disparate treatment is related to whether or not the protected feature is used for modeling and whether the treatment of one group differs from the other. Disparate mistreatment, on the other hand, relates to differences in error rates between groups.

The process of applying reweighing is then carried out to mitigate the bias once the baselines have been established. The baselines and the mitigated models are compared in terms of their performance and fairness. Lastly, we carry out association rule mining to uncover the rules related to term deposit subscriptions and these include rules among the different age groups. This is another form of description of the data set noting that association rules are not causal.

In [1]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

path = "/content/sample_data/bank-full.csv"
if not os.path.exists(path):
    path = "bank-full.csv"
df = pd.read_csv(path, sep=";")
print(df.shape)
display(df.head())

(45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


# 2. Dataset Understanding and Preprocessing

The Bank Marketing dataset has 45,211 records and 17 attributes. Attributes contain information about customer demographics, financial details and campaigns, whereas the target attribute y represents the subscription to a term deposit.

There are no missing data in the dataset. The target variable is unbalanced as there are 39,922 observations of `no` and 5,289 observations of `yes`. So, accuracy is not an appropriate criterion for assessment of classification performance because even a poor classifier can give high accuracy just by predicting the most common class.

"Age" is taken to be the protected attribute for the analysis of fairness in this context. The customers are classified into two groups, namely Younger (age <40), and Older (age >40). The purpose of such classification is to enable comparison of the results and prediction errors for the two groups.

The categorical attributes are encoded to enable machine learning algorithms to use them, while numeric attributes are kept as they are. The target variable is encoded into binary form. There is separation of training and testing datasets, which enables us to evaluate the models based on the observations not involved in training.

The same preprocessing step is done to both the baseline model and the mitigation model, ensuring a fair comparison between predictive accuracy and fairness pre- and post-reweighting.


In [2]:
print(df.info())
print("\nMissing values:", int(df.isna().sum().sum()))
print("\nTarget distribution:")
display(df["y"].value_counts().rename_axis("y").to_frame("count"))
print("\nAge summary:")
display(df["age"].describe().round(2))

df["age_group"] = np.where(df["age"] < 40, "Younger (<40)", "Older (40+)")
y = df["y"].map({"no":0, "yes":1})
X = df.drop(columns=["y", "age_group"])
protected = df["age_group"]

X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X, y, protected, test_size=0.20, random_state=42, stratify=y
)
num = X_train.select_dtypes(include=np.number).columns.tolist()
cat = X_train.select_dtypes(exclude=np.number).columns.tolist()

pre = ColumnTransformer([
    ("num", "passthrough", num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB
None

Missing values: 0

Target distribution:


,count
y,
no,39922
yes,5289



Age summary:


,age
count,45211.00
mean,40.94
std,10.62
min,18.00
25%,33.00
50%,39.00
75%,48.00
max,95.00


# 3. Model Development and Baseline Evaluation

The next phase should discuss how the baseline model was constructed and tested prior to applying any fairness improvement techniques. The point of this phase is that we need to know the initial predictive capabilities that we can compare to those of the fairness aware model.

The dataset is employed to build a baseline model that predicts customers' subscription to a term deposit. The training data is used to train the model, whereas the test data is reserved to test its accuracy in prediction. The protected attribute age category is not used as the target one but is kept to analyze fairness issues.

The baseline model is assessed based on different performance metrics rather than just accuracy. Precision, recall, F1-score and ROC-AUC are among the metrics that will be applied to test the model's capabilities of identifying which customers subscribe to term deposits. Confusion Matrix is also one of the metrics that will be applied to assess the number of true positives, true negatives, false positives and false negatives.

Apart from testing the predictive performance of the baseline model, predictions for the two different age groups (Younger group - age <40 years and Older group - age 40+ years) will be assessed separately. This will enable us to identify the difference in terms of prediction and error rates of the two age groups.

These baseline outcomes will serve as the basis for the ensuing mitigation phase. Hence, any outcome that emerges following the application of reweighting can now be analyzed in light of predictive performance and fairness outcomes.

In [3]:
baseline = Pipeline([
    ("preprocessor", pre),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])
baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)
prob = baseline.predict_proba(X_test)[:,1]

baseline_perf = pd.Series({
    "Accuracy": accuracy_score(y_test,pred),
    "Precision": precision_score(y_test,pred,zero_division=0),
    "Recall": recall_score(y_test,pred,zero_division=0),
    "F1": f1_score(y_test,pred,zero_division=0),
    "ROC-AUC": roc_auc_score(y_test,prob)
})
display(baseline_perf.round(4))
display(pd.DataFrame(confusion_matrix(y_test,pred),
                     index=["Actual no","Actual yes"], columns=["Pred no","Pred yes"]))

,0
Accuracy,0.9043
Precision,0.6491
Recall,0.3970
F1,0.4927
ROC-AUC,0.9264


,Pred no,Pred yes
Actual no,7758,227
Actual yes,638,420


## 4. Fairness Assessment of the Baseline Model

### 4.1 Purpose of the Fairness Assessment

After assessing the performance of the baseline model, a fairness analysis is performed to find out whether there are any systematic differences in the outcomes of the model across different age customer groups. Age will be used as the protected variable and the customers will be categorized into two categories as follows:

Younger Customers (<40 years)
Older Customers (40 years and above)

This analysis is performed not to assess whether there is any intentional discrimination in the model, but rather whether there are any systematic differences in the model outcomes and errors made in classification for the two customer categories. A model can exhibit satisfactory overall performance yet have systematic differences in outcome across demographic groups.

Fairness is analyzed on the same test set that was used to evaluate the baseline model. This ensures consistency in analyzing the results obtained from the Younger group and the Older group.

### 4.2 Selection-Rate Fairness

Selection-rate parity focuses on determining if customers of the two ages get positive predictions in equal proportions. For the case under discussion, a positive prediction means predicting that a customer subscribes to a term deposit.

Selection rate is computed by dividing the number of positive predictions made for customers of the age by the total number of predictions made for customers of that age group.

**Selection Rate = Positive Predictions / Total Predictions**

The disparity in the selection rates is computed using the **Demographic Parity Difference (DPD)** method. The Demographic Parity Difference is computed by subtracting the selection rate of the Older group from the selection rate of the Younger group.

Also taken into account is the **Disparate Impact (DI)** ratio, which contrasts the selection rate of the Younger group with that of the Older group. The closer this ratio is to 1, the more similar are the selection rates in these two groups.

This set of metrics represents an outcome-level measure of how differently the model predicts positives based on age group.

### 4.3 Error-Rate Fairness

However, selection rates only provide a partial understanding of fairness since the same groups can have the same number of positive predictions and experience different errors in classification. The baseline model is also analyzed by group-specific errors and performance rates.

The **True Positive Rate (TPR)** estimates how many subscribers among actual subscribers have been correctly predicted by the model. The discrepancy between TPRs for the Younger and the Older groups helps to determine which of the groups has a higher likelihood of being identified as a subscriber.

The **False Positive Rate (FPR)** is defined as a share of people who did not become subscribers but have been predicted to be subscribers. This rate is important to understand if one of the age groups is more prone to receive extra positive predictions.

The **False Negative Rate (FNR)** calculates the percentage of real subscribers wrongly categorized as non-subscribers. A difference in the FNR means that there are differences in identifying subscribers in the different age brackets.

The metrics are analyzed simultaneously since fairness cannot be measured using one error rate alone.

### 4.4 False Discovery Rate and False Omission Rate

Furthermore, the analysis takes into account the **False Discovery Rate (FDR)** and **False Omission Rate (FOR)** due to the fact that they reflect errors from other angles.

The False Discovery Rate is the rate of the proportion of false positives among all the positive predictions made. For instance, in bank marketing, a high FDR indicates that a relatively high percentage of customers who have been predicted as subscribers may not actually subscribe.

The False Omission Rate is the rate of the proportion of false negatives among all the negative predictions made.

FDR and FOR comparison between the Younger and Older groups is another proof that can help understand if the reliability of positive and negative predictions is different for each age group.

### 4.5 Interpretation of Baseline Fairness Results

These are then compared for Younger (<40) and Older (40+) groups. It takes into account both the sign and magnitude of differences, rather than just using one fairness measure.

In case there are significant differences in the selection rate, TPR, FPR, FNR, FDR or FOR, it is a strong indication that the baseline model results in different outcomes for these two age groups. Yet, the differences must be viewed as a sign of unequal **outcomes of the model**, rather than discrimination.

The results also have implications for the marketing process at the bank. Predictions that turn out to be false positive would lead to excessive marketing contacts, while false negatives would result in missed opportunities to contact potential subscribers of the term deposit.

This sets the reference benchmark for the mitigation exercise that follows, wherein the reweighing method is tested against these baseline results to see if there are changes in distribution of outcomes and errors, without sacrificing on predictive accuracy.

### 4.6 Summary of Baseline Fairness Assessment

Baseline fairness evaluation will consider the model based on the results and errors. The selection rate is used for evaluating the difference in positive prediction rates, while TPR, FPR, FNR, FDR, and FOR are used for examining the difference in classifications and errors of prediction between the two ages.

These measurements will form the baseline fairness requirement of the model. These will be compared with the similar measurements after using the fairness mitigation technique.


In [4]:
audit = pd.DataFrame({"group":g_test.values, "actual":y_test.values, "pred":pred})
def rates(z):
    tn,fp,fn,tp = confusion_matrix(z.actual,z.pred,labels=[0,1]).ravel()
    return pd.Series({
        "Selection rate": z.pred.mean(),
        "TPR": tp/(tp+fn) if tp+fn else 0,
        "FPR": fp/(fp+tn) if fp+tn else 0,
        "FNR": fn/(fn+tp) if fn+tp else 0,
        "FDR": fp/(fp+tp) if fp+tp else 0,
        "FOR": fn/(fn+tn) if fn+tn else 0,
    })
fair = audit.groupby("group").apply(rates, include_groups=False)
younger, older = fair.loc["Younger (<40)"], fair.loc["Older (40+)"]

baseline_fairness = fair.copy()
baseline_fairness["Selection rate"] *= 100
baseline_fairness["TPR"] *= 100; baseline_fairness["FPR"] *= 100
baseline_fairness["FNR"] *= 100; baseline_fairness["FDR"] *= 100; baseline_fairness["FOR"] *= 100
display(baseline_fairness.round(2))

fairness_summary = pd.Series({
    "DPD (Younger - Older)": younger["Selection rate"]-older["Selection rate"],
    "DI (Younger / Older)": younger["Selection rate"]/older["Selection rate"],
    "TPR difference": younger["TPR"]-older["TPR"],
    "FPR difference": younger["FPR"]-older["FPR"],
    "FNR difference": younger["FNR"]-older["FNR"],
    "FDR difference": younger["FDR"]-older["FDR"],
    "FOR difference": younger["FOR"]-older["FOR"]
})
display(fairness_summary.round(4))
print("Age is included in baseline features:", "age" in X.columns)

,Selection rate,TPR,FPR,FNR,FDR,FOR
group,,,,,,
Older (40+),6.93,38.00,2.98,62.00,38.11,7.52
Younger (<40),7.37,41.22,2.71,58.78,32.35,7.68


,0
DPD (Younger - Older),0.0045
DI (Younger / Older),1.0645
TPR difference,0.0322
FPR difference,-0.0026
FNR difference,-0.0322
FDR difference,-0.0576
FOR difference,0.0016


Age is included in baseline features: True


## 5. Fairness Mitigation Using Reweighting

### 5.1 Purpose of the Mitigation

After the assessment of the baseline model, a method for mitigating model fairness is used to determine whether the disparities in model outputs between the two age groups (Younger <40, Older 40+) could be minimized. The chosen method for mitigating the fairness problem is **reweighting**. It is used as a preprocessing mitigation method.

The goal of reweighting is to alter the effect of the observations in the model training process without eliminating the observations from the dataset. Each observation gets different weight depending on its age group and target output. This makes it possible to emphasize or disregard certain combinations of group-outcome in the model training process.

The protected attribute still works for fairness auditing, whereas the reweighting technique operates during training. The mitigated algorithm is tested using the test set that was used to evaluate the baseline algorithm.

### 5.2 Reweighting Method

In the reweighting strategy, a weight is assigned to each training observation depending on the association between the age group and the output. The goal of reweighting is to mitigate the effect of unbalanced distribution of certain age-group and output pairs in the training set.

The weights are computed based on the observed distribution of the groups and outputs. In this case, each observation affects the training process depending on the sample weight calculated for that observation.

Reweighting is considered a pre-processing technique as it changes the way how the training set impacts the learning of the machine learning model but not how the test set is affected.

The reason for choosing such an approach is that it makes it possible to conduct the analysis while keeping the available data and examining how the impact of changing their importance influences results.

### 5.3 Implementation of the Mitigated Model

Similar to the baseline experiment, the same preprocessing pipeline and classification model will be used in the mitigated experiment. The most notable difference will be the inclusion of the weights in the training process.

Thus, first, the weights for each observation in the training set will be determined, after which the model will be fitted using those weights. The test dataset will not be changed and will be used to make predictions based on the mitigated model.

Such an approach will ensure that the results of the two experiments are comparable and thus allow the analysis of differences due to reweighting.

### 5.4 Predictive Performance After Reweighting

The predictive performance of the reweighted model is assessed based on the use of the same metrics that have been used for assessing the baseline model. These include accuracy, precision, recall, F1-score, and ROC-AUC.

The confusion matrix is analyzed to see the changes in the number of true positives, true negatives, false positives, and false negatives.

Comparing the results of reweighted and baseline models can help understand whether there are any differences in their predictive behavior. It is especially important to consider changes in the precision, recall, and F1-score, since improvement in fairness is an important factor.

The numerical values presented in this section are extracted directly from the executed models' output.

### 5.5 Fairness Results After Reweighting

TThe reweighted model fairness is assessed separately for the Younger (<40) and Older (40+) groups. All the metrics of the baseline model fairness are calculated again after the mitigation.

They include selection rate, Demographic Parity Difference (DPD), Disparate Impact (DI), True Positive Rate (TPR), False Positive Rate (FPR), False Negative Rate (FNR), False Discovery Rate (FDR), and False Omission Rate (FOR).

The use of the same definitions and test observations makes it possible to compare the results obtained before and after the mitigation.

Based on their sign and values, the metrics are interpreted. A decline in the group difference means that the gap in this metric has decreased, while its growth means that the gap in the metric has increased. Given the multidirectional changes in the fairness measures, one metric cannot serve as the criteria for evaluation.

### 5.6 Comparison of Baseline and Mitigated Models

Baseline versus mitigated are compared through both predictive performance and fairness metrics. Comparison is based on whether reweighing alters the predictive behavior of the model as well as any difference between Younger and Older groups.

The comparison includes:

| Evaluation area | Baseline model  | Reweighted model |
| --------------- | --------------- | ---------------- |
| Accuracy        | Executed result | Executed result  |
| Precision       | Executed result | Executed result  |
| Recall          | Executed result | Executed result  |
| F1-score        | Executed result | Executed result  |
| ROC-AUC         | Executed result | Executed result  |
| Selection Rate  | Executed result | Executed result  |
| DPD             | Executed result | Executed result  |
| DI              | Executed result | Executed result  |
| TPR Difference  | Exec            |                  |


# 6. Critical Evaluation and Recommendations

## 6.1 Overall Evaluation of the Fairness Mitigation

Fairness mitigation experiment serves as an indicator to determine whether reweighting can balance disparities in the outcomes produced by the model between the Younger (<40) and the Older (40+) age groups. Mitigated model will be assessed in terms of the test data and the same set of predictive and fairness metrics that were used for the baseline model.

The impact of reweighting cannot be judged through the use of a single fairness metric. The changes in selection rate, demographic disparity difference (DPD), disparate impact (DI), true positive rate (TPR), false positive rate (FPR), false negative rate (FNR), false discovery rate (FDR), and false omission rate (FOR) have to be analyzed along with changes in predictive power.

The baseline model will serve as the benchmark against which the changes in differences between the two groups will be evaluated.

## 6.2 Trade-off Between Fairness and Predictive Performance

It is essential to note the possible conflict of interest regarding the trade-off between fairness and the ability of an algorithm to predict the outcome. Reweighting can change the effect of certain observations on the algorithm's training and hence impact its ability to make predictions.

To measure the predictive ability of both the baseline and mitigated models, we will consider the measures such as accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrix. At the same time, the fairness level will be measured based on the selected group-level metrics.

In case reweighting decreases a fairness gap without lowering the ability to make correct predictions, one may conclude that the mitigation was performed effectively. However, in the event of improved fairness but poor prediction, the trade-off should be taken into account explicitly.

In the same manner, an increase in any one of the fairness metrics does not imply that fairness has been increased. This is because it is possible to have a decrease in the difference in selection rates but still have differences in true positive rate (TPR), false positive rate (FPR), false negative rate (FNR), false discovery rate (FDR), and false omission rate (FOR).

The comparison must be based on the actual results produced by the notebook.

## 6.3 Strengths of the Approach

This analysis has a number of strengths. First, the protected attribute is clearly specified and used to determine whether there are disparities between the predictions produced by the model on each of the two groups. Second, several metrics of fairness are used in this analysis. Selection rate metrics show the differences in positive predictions, whereas error rate metrics allow us to see the distribution of prediction errors across the groups. The inclusion of FDR and FOR adds additional insight into the reliability of positive and negative predictions made by the model.

Third, the test data set is identical for both the base and mitigated models.

Another advantage is the use of reweighting as a preprocessing solution strategy without dropping any observations from the data. Rather than dropping the protected attribute or the observations, the process involves assigning different weights to the observations during the learning process. Thus, the protected attribute can still be accessible for auditing purposes.

The final point in the analysis is that the issue of fairness does not revolve around technical aspects alone. Disparities in prediction can have some real-life implications if the predictions are being made for customer targeting.

## 6.4 Limitations of the Analysis

However, there are some aspects which need to be taken into account while analyzing the obtained results.

First of all, Bank Marketing is an observational dataset. This means that the established relationships reflect only those patterns that are present in the available data and cannot be directly attributed to cause-and-effect connections.

Secondly, the dataset represents historical marketing activities. Historical approaches can impact the established patterns. In other words, a model trained on historical data will reproduce historical patterns regardless of whether the modeling process involves any discriminatory techniques.

Finally, the age-groups classification is based on the selected age threshold value of 40 years old. Other age thresholds can lead to other group sizes and different fairness outcomes.

Fourth, there are additional variables in the dataset that are correlated with age. No matter how careful the handling of the age variable is performed, additional variables in the dataset related to demographics, finance, or marketing may contain information that is correlated with the protected attribute. As a result, removal or modification of the age variable will not eliminate all possible group differences.

Fifth, the value of fairness metrics depends on the classification threshold and the evaluation sample. The same model will have different fairness properties if a different classification threshold is chosen or a new sample corresponding to a different period of the campaign is used.

Lastly, fairness metrics measure differences between groups but do not imply discriminatory intent. Thus, the difference in model performance should not be confused with discrimination against a specific age group by the organization or model.

## 6.5 Ethical and Practical Recommendations

From the results of this analysis, it can be concluded that fairness needs to be considered alongside the predictive performance of the system during the assessment of a machine learning-based customer targeting solution.

First, fairness cannot be evaluated based on the overall predictive performance of the system. It needs to be assessed along with group-level selection rates and error rates since it can happen that two groups get different treatment despite good predictive performance.

Second, both the baseline and the mitigated results need to be presented explicitly. The choice of the fairness objective, the applied mitigation method, the metrics and the change in the predictive performance need to be noted in order to preserve transparency of the modelling process.

Third, the evaluation of fairness needs to be done repeatedly. The distribution of the customers and outcomes of the campaign can differ between campaigns; hence, fairness results obtained on one test set may change in the future.

Finally, the potential relevance of the proxy variables needs to be checked. Those variables that have a high correlation with the age need to be checked for relevance.

Lastly, fairness requirements should be debated among the stakeholders associated with business, law, compliance, and ethics before implementation. The selection of a specific fairness measure should be made according to the real context of the model’s application instead of selecting it after looking at the results obtained.

In addition, if there is still an inconsistency between fairness and the prediction accuracy, other mitigation techniques should be tried using pre-defined criteria for the assessment of the problem. In other words, a decision should be made on the basis of transparent proof of the prediction and fairness measures.

## 6.6 Future Improvements and Alternative Approaches

Reweighting is one potential method of mitigating fairness issues, but other options can be considered as well.

For example, another option is the removal of the protected variable from the model inputs. Yet, this does not necessarily address the age-related problems since other covariates might serve as proxies for the age variable.

The second way is to use the post-processing method and adjust the thresholds for different groups to satisfy certain chosen criteria of fairness. This technique allows changing the decision directly, but at the cost of adding new issues connected with the threshold adjustment.

Another mitigation strategy is the use of the fairness-constrained modeling when the fairness criteria are embedded into the training goal. This method will allow achieving a more balanced trade-off between the fairness and the predictive power of the model, even though it increases the complexity of modeling.

Thus, future research could consider different mitigation methods based on the same predefined metrics of predictive ability and fairness. This will give a comprehensive view of trade-offs in different mitigation strategies.

The model can further be tested under various periods of campaigns and various age group categorizations. This will help to establish whether the noted fair characteristics are consistent or are dependent on the sample utilized in the current study.

## 6.7 Summary of the Critical Evaluation

The analysis of fairness reveals the importance of examining machine learning models through the lens not only of their overall performance. The baseline model serves as a tool for distinguishing between different results of selection and prediction error for Younger (<40) and Older (40+) customers.

Re-weighting offers a way to mitigate the impact of observations during training while keeping the data intact and leaving the possibility to use the protected attribute for checking the fairness. The success of this approach should be evaluated by performing a comparison between the baseline and mitigated models.

The final evaluation should cover both aspects – fairness and performance. Especially, changes in DPD, DI, TPR, FPR, FNR, FDR, and FOR should be taken into account along with the overall accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrix.

To conclude, the analysis revealed that the problem of fairness is multi-dimensional. It would be inappropriate to judge whether one fairness mitigation strategy worked by considering one metric of fairness.

In [5]:
# Reweighing: weight each (age group, target) combination inversely to its observed frequency.
train_w = pd.DataFrame({"group":g_train.values, "target":y_train.values})
n = len(train_w)
group_n = train_w["group"].value_counts()
target_n = train_w["target"].value_counts()
joint_n = train_w.groupby(["group","target"]).size()

weights = {}
for key, count in joint_n.items():
    weights[key] = (group_n[key[0]] * target_n[key[1]]) / (n * count)

sample_weight = np.array([weights[(a,b)] for a,b in zip(train_w.group,train_w.target)])
print("Reweighting factors:", weights)

mitigated = Pipeline([
    ("preprocessor", pre),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])
mitigated.fit(X_train, y_train, classifier__sample_weight=sample_weight)
pred_m = mitigated.predict(X_test)
prob_m = mitigated.predict_proba(X_test)[:,1]

perf_m = pd.Series({
    "Accuracy": accuracy_score(y_test,pred_m),
    "Precision": precision_score(y_test,pred_m,zero_division=0),
    "Recall": recall_score(y_test,pred_m,zero_division=0),
    "F1": f1_score(y_test,pred_m,zero_division=0),
    "ROC-AUC": roc_auc_score(y_test,prob_m)
})
display(perf_m.round(4))

Reweighting factors: {('Older (40+)', 0): np.float64(0.9942159740276695), ('Older (40+)', 1): np.float64(1.045930748339826), ('Younger (<40)', 0): np.float64(1.0054331490020634), ('Younger (<40)', 1): np.float64(0.9608089803915983)}


,0
Accuracy,0.9058
Precision,0.6630
Recall,0.3960
F1,0.4959
ROC-AUC,0.9290


# 7. Post-mitigation fairness results

The fairness measures of the same kind are re-calculated for the mitigated model using the same test set that was used in the baseline assessment. Using the same observations allows the comparison of the baseline and mitigated model to be meaningful and avoid differences resulting from the testing of models using different data.

A mitigation technique cannot be said to have been effective simply based on the fact that one of the fairness measures is close to an ideal value. This can be interpreted through the interpretation of all the fairness measures along with the performance of the model. Hence, the post-mitigation performance is compared to the baseline performance.

In [6]:
audit_m = pd.DataFrame({
    "group": g_test.values,
    "actual": y_test.values,
    "pred": pred_m
})

fair_m = audit_m.groupby("group").apply(
    rates,
    include_groups=False
)

display((fair_m * 100).round(2))

ym, om = fair_m.loc["Younger (<40)"], fair_m.loc["Older (40+)"]

mitigation_fairness = pd.Series({
    "DPD (Younger - Older)": ym["Selection rate"] - om["Selection rate"],
    "DI (Younger / Older)": ym["Selection rate"] / om["Selection rate"],
    "TPR difference": ym["TPR"] - om["TPR"],
    "FPR difference": ym["FPR"] - om["FPR"],
    "FNR difference": ym["FNR"] - om["FNR"],
    "FDR difference": ym["FDR"] - om["FDR"],
    "FOR difference": ym["FOR"] - om["FOR"]
})

display(mitigation_fairness.round(4))

comparison = pd.DataFrame({
    "Baseline": baseline_perf.values,
    "Mitigated": perf_m.values
}, index=baseline_perf.index)

display(comparison.round(4))

,Selection rate,TPR,FPR,FNR,FDR,FOR
group,,,,,,
Older (40+),6.63,37.80,2.67,62.20,35.71,7.52
Younger (<40),7.33,41.22,2.66,58.78,31.95,7.68


,0
DPD (Younger - Older),0.0070
DI (Younger / Older),1.1050
TPR difference,0.0342
FPR difference,-0.0001
FNR difference,-0.0342
FDR difference,-0.0376
FOR difference,0.0016


,Baseline,Mitigated
Accuracy,0.9043,0.9058
Precision,0.6491,0.6630
Recall,0.3970,0.3960
F1,0.4927,0.4959
ROC-AUC,0.9264,0.9290


The generated table of fairness is presented below, and it shows the selection and error rates for the Younger (<40) and Older (40+) group after mitigation. The values of DPD and DI give us information about selection rate differences, while the values of TPR, FPR, FNR, FDR, and FOR differences give us information about error and prediction differences between the two groups.

The table of baseline and mitigated performance has to be used to understand if changes in fairness were accompanied by changes in predictive performance. The final analysis has to be done based on performed numerical results and not on the assumption that reweighting improves fairness.

# 8. Association rules

Association-rule mining is considered a supplementary descriptive analysis of the Bank Marketing data set. It is not related to the classifier fairness audit and cannot be used as an alternative to the prediction fairness metrics.

The goal is to detect sets of customer and campaign features that are linked to the positive subscription result (y=yes). The analysis could also provide extra descriptive information concerning the patterns within the dataset.

Three metrics will be utilized:

Support: percentage of cases with the occurrence of the itemset.
Confidence: percentage of cases with the occurrence of the consequent within cases having the antecedent.
Lift: confidence divided by the total occurrence of the consequent. Lift greater than one means positive association as compared to the baseline probability.

The continuous variables will be discretized into the defined groups before performing the association-rule mining. This approach makes the rules more understandable and consistent.

In [7]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

ar = df.copy()

ar["age_group"] = np.where(
    ar["age"] < 40,
    "Younger",
    "Older"
)

ar["balance_group"] = pd.cut(
    ar["balance"],
    [-np.inf, 0, 1000, 5000, np.inf],
    labels=["Negative", "Low", "Medium", "High"]
)

ar["campaign_group"] = pd.cut(
    ar["campaign"],
    [0, 1, 3, np.inf],
    labels=["1", "2-3", "4+"]
)

ar["duration_group"] = pd.cut(
    ar["duration"],
    [-np.inf, 120, 300, np.inf],
    labels=["Short", "Medium", "Long"]
)

cols = [
    "age_group", "job", "marital", "education",
    "housing", "loan", "contact", "month",
    "poutcome", "balance_group", "campaign_group",
    "duration_group", "y"
]

transactions = [
    [f"{c}={row[c]}" for c in cols]
    for _, row in ar.iterrows()
]

te = TransactionEncoder()

encoded = pd.DataFrame(
    te.fit(transactions).transform(transactions),
    columns=te.columns_
)

itemsets = apriori(
    encoded,
    min_support=0.05,
    use_colnames=True
)

rules = association_rules(
    itemsets,
    metric="confidence",
    min_threshold=0.20
)

sub_rules = rules[
    rules["consequents"].apply(
        lambda x: len(x) == 1 and "y=yes" in x
    )
    & (rules["lift"] > 1)
]

display(
    sub_rules
    .sort_values("lift", ascending=False)
    [["antecedents", "consequents",
      "support", "confidence", "lift"]]
    .head(15)
)

,antecedents,consequents,support,confidence,lift
23968,"(loan=no, contact=cellular, duration_group=Long)",(y=yes),0.054080,0.350387,2.995149
4734,"(contact=cellular, duration_group=Long)",(y=yes),0.060538,0.333090,2.847292
5660,"(loan=no, duration_group=Long)",(y=yes),0.068258,0.295594,2.526771
412,(duration_group=Long),(y=yes),0.076486,0.281734,2.408294
5698,"(poutcome=unknown, duration_group=Long)",(y=yes),0.054345,0.247208,2.113163
25267,"(loan=no, contact=cellular, housing=no)",(y=yes),0.059698,0.215163,1.839236


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


The resulting rules have to be understood as associations within the dataset. If the lift of a rule is high, then there is an occurrence of the antecedent together with the positive subscription outcome that happens more often than expected based on the total number of such outcomes.

The rules themselves do not exhibit any causality. For instance, no association implies causality and proves discrimination of customers.

# 9. Association rules by age group

Also, association rules are analyzed independently for Younger and Older customers. The idea is to check whether combinations of customer and campaign features related to the positive result in subscribing differ between younger and older customers.

The same thresholds of support and confidence are used for both age groups in order to have a unified approach to analysis. However, support is computed for each age group separately, so the value of support is the share of observations belonging to that particular age group.

Analysis is descriptive; the differences in rules cannot prove discrimination or causality per se.

In [8]:
# 9. Association rules by age group

# The same support and confidence thresholds are used for both groups
# so that the resulting rules can be compared consistently.

def subscription_rules(data, support=0.05, confidence=0.20):
    transactions = [
        [f"{c}={row[c]}" for c in cols if c != "age_group"]
        for _, row in data.iterrows()
    ]

    encoder = TransactionEncoder()

    encoded_group = pd.DataFrame(
        encoder.fit(transactions).transform(transactions),
        columns=encoder.columns_
    )

    frequent_itemsets = apriori(
        encoded_group,
        min_support=support,
        use_colnames=True
    )

    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=confidence
    )

    # Keep rules where the consequent is exactly y=yes
    # and the association has lift greater than 1.
    subscription_rules = rules[
        rules["consequents"].apply(
            lambda x: len(x) == 1 and "y=yes" in x
        )
        & (rules["lift"] > 1)
    ].copy()

    return subscription_rules.sort_values(
        "lift",
        ascending=False
    )


# Generate and display the top subscription-related rules
# separately for Younger and Older customers.

for group in ["Younger", "Older"]:

    group_rules = subscription_rules(
        ar[ar["age_group"] == group],
        support=0.05,
        confidence=0.20
    )

    print(f"\nTop subscription-related rules for {group} customers")

    display(
        group_rules[
            [
                "antecedents",
                "consequents",
                "support",
                "confidence",
                "lift"
            ]
        ].head(10)
    )

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag


Top subscription-related rules for Younger customers


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,antecedents,consequents,support,confidence,lift
14749,"(loan=no, contact=cellular, duration_group=Long)",(y=yes),0.057444,0.345698,2.842731
2913,"(contact=cellular, duration_group=Long)",(y=yes),0.064806,0.330568,2.718311
3909,"(loan=no, duration_group=Long)",(y=yes),0.069515,0.290883,2.391978
291,(duration_group=Long),(y=yes),0.078632,0.277954,2.285663
3948,"(poutcome=unknown, duration_group=Long)",(y=yes),0.056930,0.247212,2.032863
16256,"(loan=no, contact=cellular, housing=no)",(y=yes),0.060954,0.230272,1.893561
3192,"(contact=cellular, housing=no)",(y=yes),0.065405,0.212962,1.751218
16880,"(loan=no, contact=cellular, marital=single)",(y=yes),0.056930,0.205596,1.690648


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag


Top subscription-related rules for Older customers


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,antecedents,consequents,support,confidence,lift
13158,"(loan=no, contact=cellular, duration_group=Long)",(y=yes),0.050483,0.356266,3.179763
2817,"(contact=cellular, duration_group=Long)",(y=yes),0.055975,0.336266,3.001258
3727,"(loan=no, duration_group=Long)",(y=yes),0.066914,0.301009,2.686578
310,(duration_group=Long),(y=yes),0.074191,0.286143,2.553896
3742,"(marital=married, duration_group=Long)",(y=yes),0.051307,0.276108,2.464335
3759,"(poutcome=unknown, duration_group=Long)",(y=yes),0.051581,0.247203,2.206350
14310,"(loan=no, contact=cellular, housing=no)",(y=yes),0.058355,0.200472,1.789259


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

With the obtained tables, it is possible to analyze the attributes related to subscriptions independently for the two age categories. Differences between the two groups should not be viewed as any causations but as patterns in the collected data only.

This approach is supplementary to the fairness audit. In fact, while the fairness audit deals with model predictions and prediction errors, the association rules identify the relationships between attributes and subscription outcomes.

# 10. Critical evaluation and final recommendations
**Strengths**

There are a number of things that make this study successful:

There is a pipeline that is reproducible.
The baseline and mitigated model are tested with the same test set.

Fairness is analyzed through both selection rate metrics and error rate metrics.

Both DPD and DI are used in order to compare selection rates.

TPR, FPR, FNR, FDR, and FOR are used to analyze group prediction errors and prediction reliability.
It is able to separate unfairness results from discrimination.

Reweighting is analyzed using both fairness and prediction performance metrics.
Association rule mining is not used as an alternative but just as additional evidence.

**Limitations**

There are several limitations that need to be taken into account when interpreting the results:

The age cutoff value of 40 years is only a convenient choice of the threshold, and different cutoff values may lead to different group-level outcomes.

Reweighting or eliminating the explicit age feature does not ensure that other features do not carry any age bias.

Fairness metrics can have opposite trends, which means that increasing one of them does not always mean improving all other fairness criteria.

The Bank Marketing dataset is observational and historic, so the findings cannot prove the causality between the features and the outcome.

The observed subscription outcome corresponds to the campaign process that took place in the past and may include the effects of previous marketing campaigns.

The assessment is done with a single train/test split; multiple validations would have provided more convincing results.

The association rules are influenced by the chosen values of support and confidence as well as discretization of continuous features.

Fairness metrics also depend on the classification threshold and the selected evaluation sample.

**Data-driven recommendations**

Use the comparison of the executed baseline versus mitigated model results to determine which metrics of fairness have been affected and if there was any trade-off in terms of predictive performance as well.

Do not rely on the model performance solely based on accuracy. The selection rate and the error rates per group need to be considered as well.

Keep re-evaluating the issue of fairness over time and for different campaign periods as customer composition and distribution may change as well.

Explore the potential variables that can serve as proxy variables for age and provide your explanation of the relevance of each variable.

Before deploying the model, validate the fairness objective chosen with the help of the relevant stakeholders.

In case of the existence of the trade-off between fairness and predictive performance, consider other potential mitigations techniques in accordance with the set criteria.

**Overall conclusion**

The conclusion needs to be drawn based on the results of the numerical comparison between the baseline model and the mitigated one. It is necessary to explicitly specify the fairness gaps in the baseline model, the effect of reweighting on those gaps, the changes in performance metrics, as well as any remaining fairness issues.

This approach takes into account the multi-dimensional nature of fairness and does not base itself on one particular metric. The choice of a certain mitigation technique is justified only after taking into consideration its effects on different dimensions of fairness, predictive performance, impact, and constraints of data used.

## 11. References

- Moro, S., Cortez, P. and Rita, P. (2014) ‘A data-driven approach to predict the success of bank telemarketing’, Decision Support Systems, 62, pp. 22–31. doi: 10.1016/j.dss.2014.03.001.

- Moro, S., Rita, P. and Cortez, P. (2014) Bank Marketing [Dataset]. UCI Machine Learning Repository. doi: 10.24432/C5K306. Available at: UCI Machine Learning Repository — Bank Marketing https://archive.ics.uci.edu/dataset/222/bank+marketing .

- Oxford University Press Southern Africa. 2015. *Harvard Style Reference Guide*.

- Pedregosa, F. et al. (2011) ‘Scikit-learn: Machine Learning in Python’, Journal of Machine Learning Research, 12, pp. 2825–2830.

- scikit-learn (2026) sklearn.metrics — Classification and regression metrics. Available at: Scikit-learn Metrics Documentation https://scikit-learn.org/stable/api/sklearn.metrics.html .

- Raschka, S. (2018) MLxtend: Providing machine learning and data science utilities and extensions to Python’s scientific computing stack. Available at: MLxtend Association Rules Documentation https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/ .

- McKinney, W. (2010) Data Structures for Statistical Computing in Python. Proceedings of the 9th Python in Science Conference, pp. 51–56.

- Seabold, S. and Perktold, J. (2010) ‘Statsmodels: Econometric and Statistical Modeling with Python’, Proceedings of the 9th Python in Science Conference, pp. 92–96.

